In [0]:
BASE_DIR = '/Volumes/sales/landing/volumes/Orders/'
CHECKPOINT_DIR = '/Volumes/sales/landing/volumes/checkpoint_dir'
CHECKPOINT_LOCATION = f'{CHECKPOINT_DIR}/Orders'
CATALOG = 'SALES'
SCHEMA = 'BRONZE'
TARGET_TABLE = 'ORDERS'
print(f'Checkpoint Location: {CHECKPOINT_LOCATION}')
print(f'Base Location: {BASE_DIR}')

In [0]:
orders_schema = """order_id STRING, 
                 customer_id STRING,
                 order_status STRING,
                 order_purchase_timestamp TIMESTAMP,
                 order_approved_at TIMESTAMP,
                 order_delivered_carrier_date TIMESTAMP,
                 order_delivered_customer_date TIMESTAMP,
                 order_estimated_delivery_date TIMESTAMP
                 """

In [0]:
orders_df = (spark.read.format('csv').option('header','true').schema(orders_schema).load(f'{BASE_DIR}')
           )
display(orders_df.limit(3))
orders_df.printSchema()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

orders_df = (
            spark.readStream.format('cloudFiles')
                 .option('cloudFiles.format', 'csv')
                 .option('cloudFiles.schemaEvolutionMode', 'rescue')
                 .option('header', 'true')
                 .schema(orders_schema)
                 .load(f'{BASE_DIR}')
        )
orders_trans_df = orders_df.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "_metadata.file_name",
    "_metadata.file_path",).withColumn('load_timestamp', current_timestamp())

In [0]:
qry = (
    orders_trans_df.writeStream.format('delta')
    .outputMode('append')
    .option('checkpointLocation', CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable(f'{CATALOG}.{SCHEMA}.{TARGET_TABLE}')
)
qry.awaitTermination()